# 00a_environment_preflight

**Task ID:** `ENVIRONMENT_PREFLIGHT`  
**Runner:** `experiments.scripts.environment_preflight`

## Scientific responsibility

Validate environment, hardware, package/model availability and data/cache paths.

## Required artifacts

- None

## Produced artifacts

- `environment.profile`
- `environment.preflight_receipt`

This notebook is a thin orchestrator. Model execution, metrics, schemas and
artifact validation live in versioned Python modules. It must run in a clean
kernel and receives its frozen profile/provenance through executor-controlled
environment variables.


In [ ]:
# Executor-controlled configuration; fail closed when run directly.
import json
import os
from pathlib import Path

NOTEBOOK_ID = '00a_environment_preflight'
TASK_ID = 'ENVIRONMENT_PREFLIGHT'
RUNNER_MODULE = 'experiments.scripts.environment_preflight'
REQUIRED_ENV = (
    "MLS_BENCHMARK_PROFILE",
    "MLS_RUN_NAMESPACE",
    "MLS_NOTEBOOK_ID",
    "MLS_CONFIG_SHA256",
    "MLS_ARTIFACT_ROOT",
    "MLS_RUN_ID",
    "MLS_SOURCE_SHA256",
    "MLS_DEPENDENCY_STATE_SHA256",
    "MLS_EVIDENCE_STATUS",
)
missing = [name for name in REQUIRED_ENV if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Missing executor-controlled variables: {missing}")
if os.environ["MLS_NOTEBOOK_ID"] != NOTEBOOK_ID:
    raise RuntimeError("Notebook ID does not match executor contract")

RUN_CONTEXT = {name: os.environ[name] for name in REQUIRED_ENV}
print(json.dumps({"task_id": TASK_ID, **RUN_CONTEXT}, indent=2))


In [ ]:
# Delegate all scientific computation to the registered runner.
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    RUNNER_MODULE,
    "--profile",
    RUN_CONTEXT["MLS_BENCHMARK_PROFILE"],
    "--run-namespace",
    RUN_CONTEXT["MLS_RUN_NAMESPACE"],
    "--evidence-status",
    RUN_CONTEXT["MLS_EVIDENCE_STATUS"],
    "--artifact-root",
    RUN_CONTEXT["MLS_ARTIFACT_ROOT"],
    "--notebook-id",
    NOTEBOOK_ID,
    "--run-id",
    RUN_CONTEXT["MLS_RUN_ID"],
    "--source-sha256",
    RUN_CONTEXT["MLS_SOURCE_SHA256"],
    "--config-sha256",
    RUN_CONTEXT["MLS_CONFIG_SHA256"],
    "--dependency-state-sha256",
    RUN_CONTEXT["MLS_DEPENDENCY_STATE_SHA256"],
]
completed = subprocess.run(command, check=False, text=True)
if completed.returncode != 0:
    raise RuntimeError(f"Runner failed with exit code {completed.returncode}")


In [ ]:
# Revalidate evidence before the executor can promote this run.
from benchmarks.scripts.execute_all_notebooks import validate_produced_artifacts

validation_context = {
    "run_id": RUN_CONTEXT["MLS_RUN_ID"],
    "status": RUN_CONTEXT["MLS_EVIDENCE_STATUS"],
    "source_sha256": RUN_CONTEXT["MLS_SOURCE_SHA256"],
    "config_sha256": RUN_CONTEXT["MLS_CONFIG_SHA256"],
    "dependency_state_sha256": RUN_CONTEXT["MLS_DEPENDENCY_STATE_SHA256"],
}
entry_contract = {"expected_schema_version": 1, "group": "00", "id": "00a_environment_preflight", "network_policy": "none", "path": "experiments/notebooks/00a_environment_preflight.ipynb", "produces": ["environment.profile", "environment.preflight_receipt"], "profiles": ["smoke", "full"], "requires": [], "responsibility": "Validate environment, hardware, package/model availability and data/cache paths.", "runner": "experiments.scripts.environment_preflight", "task_id": "ENVIRONMENT_PREFLIGHT", "timeout_seconds": 300}
validation_errors = validate_produced_artifacts(
    entry_contract,
    Path(RUN_CONTEXT["MLS_ARTIFACT_ROOT"]),
    RUN_CONTEXT["MLS_BENCHMARK_PROFILE"],
    RUN_CONTEXT["MLS_RUN_NAMESPACE"],
    validation_context,
)
if validation_errors:
    raise RuntimeError("Produced evidence failed validation:\n- " + "\n- ".join(validation_errors))
print(f"Validated {len(entry_contract['produces'])} artifact contract(s)")
